In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
spark.conf.set("spark.sql.session.timeZone", "UTC")

In [0]:
base_path = dbutils.fs.ls("abfss://bronze@staccpakwheels.dfs.core.windows.net/pakwheelsgooddata")
brands = ["Toyota", "Honda", "Suzuki", "MG", "Changan", "Haval", "JAC", "Daihatsu"]

In [0]:
base_path = "abfss://bronze@staccpakwheels.dfs.core.windows.net/pakwheelsgooddata"

folders = dbutils.fs.ls(base_path)

dfs = []

for fo in folders:
    brand = fo.name.replace("scraped_data_", "").replace("/", "")
    
    df = (
        spark.read
        .format("parquet")
        .option("mergeSchema", "true")
        .load(fo.path)   # ✅ THIS is the key
    )
    
    dfs.append(df)

In [0]:

bronze_df = dfs[0]
for d in dfs[1:]:
    bronze_df = bronze_df.unionByName(d, allowMissingColumns=True)


In [0]:
# bronze_df.show(1,False)

In [0]:
def normalize_text(col):
    return F.initcap(F.trim(F.regexp_replace(col, "\\s+", " ")))

In [0]:
silver_df = (
    bronze_df
    .withColumn("MakeModel", normalize_text(F.col("MakeModel")))
    .withColumn("City", normalize_text(F.col("City")))
    .withColumn("Colour", normalize_text(F.col("Colour")))
    .withColumn("FuelType", normalize_text(F.col("FuelType")))
    .withColumn("Transmission", normalize_text(F.col("Transmission")))
    .withColumn("ManufactureType", normalize_text(F.col("ManufactureType")))
    .withColumn("CarType", normalize_text(F.col("CarType")))
)

In [0]:
silver_df = (
    silver_df
    .withColumn("Year", F.col("Year").cast("int"))
    .withColumn("Mileage", F.col("Mileage").cast("int"))
    .withColumn("EngineCapacity", F.col("EngineCapacity").cast("int"))
    .withColumn("PostLastUpdatedat", F.to_date("PostLastUpdatedat"))
    
)




In [0]:
premium_features = [
    "AlloyWheels",
    "FrontFogLights",
    "DRLs",
    "ABS",
    "AirBags",
    "AirConditioning",
    "AlloyRims",
    "InfotainmentSystem",
    "FrontSpeakers",
    "RearSpeakers",
    "PaddleShifters",
    "TractionControl",
    "RearCamera",
    "ImmobilizerKey",
    "PowerLocks",
    "PowerSteering",
    "CruiseControl",
    "SteeringSwitches",
    "FrontCamera",
    "CoolBox"]

In [0]:

allowed_values = ["0", "1", "true", "false"]
bad_premium_condition = None

for c in premium_features:
    col_bad = (
        F.col(c).isNotNull() &
        (~F.lower(F.col(c).cast("string")).isin(allowed_values))
    )
    bad_premium_condition = col_bad if bad_premium_condition is None else (bad_premium_condition | col_bad)

bad_records_df = silver_df.filter(bad_premium_condition)
good_records_df = silver_df.filter(~bad_premium_condition)


In [0]:
# ## Data quality for premium Features:
premium_features_for_Qual = [
    "AlloyWheels",
    "FrontFogLights",
    "DRLs",
    "ABS",
    "AirBags",
    "AirConditioning",
    "AlloyRims",
    "InfotainmentSystem",
    "FrontSpeakers",
    "RearSpeakers",
    "PaddleShifters",
    "TractionControl",
    "RearCamera",
    "ImmobilizerKey",
    "PowerLocks",
    "PowerSteering",
    "CruiseControl"]
    
for col_name in premium_features_for_Qual:
    silver_df_good = good_records_df.withColumn(
        col_name,
        F.when(F.col("PriceCategory") == "Premium", F.lit(1))
         .otherwise(F.col(col_name).cast("int")) 
    )



In [0]:
mid_features = [
    "AlloyWheels",
    "FrontFogLights",
    "AirConditioning",
    "AlloyRims",
    "InfotainmentSystem",
    "FrontSpeakers",
    "RearCamera",
    "PowerLocks",
    "PowerSteering",
    
]

for col_name in mid_features:
    silver_df_good = silver_df_good.withColumn(
        col_name,
        F.when(F.col("PriceCategory") == "Mid", F.lit(1))
         .otherwise(F.col(col_name))
    )

In [0]:
# silver_df.show(1,False)

In [0]:
#silver_df.createOrReplaceTempView("silver_df")

In [0]:
#%sql
#select PriceCategory, count(*) as countOfPriceCat from silver_df group by PriceCategory order by PriceCategory


In [0]:
# silver_df_good = (
#     silver_df_good
#     .withColumn(
#         "DemandLacs",
#         F.regexp_extract(F.lower(F.col("Demand")), r"(\d+)", 1).cast("int")
#     )
#     .withColumn("DemandPKR", F.col("DemandLacs") * F.lit(100000))
# )


# silver_df_good = silver_df_good.withColumn(
#     "DemandLacs",
#     F.when(
#         F.col("Demand").isNull() | (F.trim(F.col("Demand")) == ""),
#         None
#     ).otherwise(
#         F.regexp_extract(
#             F.lower(F.col("Demand")),
#             r"(\d+)",
#             1
#         ).cast("int")
#     )
    
# )

# silver_df_good = silver_df_good.withColumn("DemandPKR", F.col("DemandLacs") * F.lit(100000))


In [0]:

silver_df_good = silver_df_good.withColumn(
    "SafetyScore",
    (
        F.col("ABS").cast("int") +
        F.col("AirBags").cast("int") +
        F.col("TractionControl").cast("int") +
        F.col("RearCamera").cast("int") +
        F.col("ImmobilizerKey").cast("int")
    ).cast("double")
)


In [0]:
silver_df_good = silver_df_good.withColumn(
    "ComfortScore",
    (
        F.col("AirConditioning").cast("int") +
        F.col("PowerSteering").cast("int") +
        F.col("CruiseControl").cast("int") +
        F.col("CoolBox").cast("int") +
        F.col("SunRoof").cast("int")
    ).cast("double")
)

In [0]:
silver_df_good = silver_df_good.withColumn(
    "TechScore",
    (
        F.col("InfotainmentSystem").cast("int") +
        F.col("SteeringSwitches").cast("int") +
        F.col("PaddleShifters").cast("int") +
        F.col("FrontCamera").cast("int") +
        F.col("RearCamera").cast("int")
    ).cast("double")
)

In [0]:
for c in premium_features:
    silver_df_good = silver_df_good.withColumn(
        c,
        F.when(F.col(c) == 1, F.lit(True)).otherwise(F.lit(False))
    )

In [0]:
# silver_df_good = (
#     silver_df_good
#     .withColumn("InvalidYearFlag",
#         (F.col("Year") < 1990) | (F.col("Year") > F.year(F.current_date()))
#     )
#     .withColumn("InvalidMileageFlag", F.col("Mileage") <= 0)
#     .withColumn(
#         "MissingCriticalInfo",
#         F.when(
#             F.col("Mileage").isNull() |
#             F.col("Year").isNull() |
#             F.col("DemandPKR").isNull(),
#             1
#         ).otherwise(0)
#     )
# )

In [0]:
# silver_df_good.show(1,False)

In [0]:
# silver_df_good.printSchema()

In [0]:
base_silver_path = "abfss://silver@staccpakwheels.dfs.core.windows.net/pakwheels/cardata"


(
    silver_df_good
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .save(base_silver_path)
)
